
# サンプリング波形の異常検知 設計仕様書（第3版：複数サイクル窓→調和回帰→1周期特徴量）

## 1. 文書の目的

本書は、1本センサのサンプリング波形に対して、正常状態からの変化を異常として検知するための再現可能な設計仕様を定義するものである。

対象データは、各観測点ごとにセンサ値、時刻、カム角度 $0\sim360$ 度を持つ。  
1サイクルあたりの観測点数は少なく、典型的には 9〜12 点程度である。  
実運用では 1 サイクル単位で見るだけでなく、10 サイクル以上のまとまりで安定に見たい場面がある。

第3版では、ここを明確に修正する。

**判定単位は任意の連続 $W$ サイクル窓とし、その窓に含まれる全観測点をまとめて調和回帰し、得られた 1 周期の代表波形に対して、1サイクル版と同じ考え方で特徴量を計算する。**

つまり、第2版のように

「1サイクルごとに特徴量を出してから窓で平均する」

のではなく、第3版では

「窓内の全観測点をまとめて 1 本の周期モデルを作り、そのモデルから 1 本の代表周期波形を復元し、その波形から特徴量を出す」

という構造を採る。

---

## 2. この版で採用する考え方

### 2.1 判定単位は窓そのもの

窓長を $W$、ストライドを $S$ とする。  
窓 $h$ に含まれるサイクル集合を

$$
I_h = \{(h-1)S+1,\; (h-1)S+2,\; \dots,\; (h-1)S+W\}
$$

とする。

第3版では、窓 $h$ は単なる「1サイクル特徴量の集約先」ではない。  
**窓 $h$ 自体が、1つの解析対象サンプルである。**

### 2.2 窓内の全観測点をまとめて使う

窓 $h$ に含まれる全観測点を

$$
\{(\theta_{hm}, y_{hm}, t_{hm})\}_{m=1}^{N_h}
$$

と書く。ここで $N_h$ は窓 $h$ に含まれる総観測点数であり、

$$
N_h = \sum_{i \in I_h} n_i
$$

である。

このとき角度 $\theta_{hm}$ は、各サイクルで 0〜360 度に折り返した角度を使う。  
つまり、10サイクル窓であっても、角度は 0〜360 度の 1 周期上に重ねて扱う。

### 2.3 窓ごとに調和回帰する

窓ごとに調和回帰を行い、その窓に対して 1 本の周期モデルを作る。  
このモデルは、窓内の複数サイクルの観測を平均化した代表周期波形とみなせる。

### 2.4 補間後は 1サイクル版と同じように特徴量を出す

窓ごとの調和回帰モデルから、密な角度グリッド上の代表波形を復元する。  
その後は、1サイクル版と同じ考え方で、上下ずれ、振幅、位相、周波数配分、残差異常などの特徴量を計算する。

### 2.5 $W=1$ なら 1サイクル版に一致する

窓長 $W=1$ のとき、窓に含まれる観測点は 1 サイクルぶんだけである。  
したがって、第3版はそのまま 1サイクル版と一致する。

---

## 3. 全体の処理の流れ

処理の流れは次の 5 段である。

1. 連続データをサイクルに分割する。  
2. fit / threshold / test にサイクル列として分割する。  
3. 各群の内部で、連続 $W$ サイクル窓を作る。  
4. 各窓について、窓内の全観測点をまとめて調和回帰し、代表周期波形を補間する。  
5. その代表周期波形と残差から特徴量を計算し、窓単位の異常判定を行う。

重要なのは、**窓を作ってから調和回帰する**点である。  
1サイクルごとに別々に回帰してから平均するのではない。

---

## 4. 用語と記号

### 4.1 サイクルと観測

サイクル番号を $i=1,2,\dots,N$ とする。  
サイクル $i$ の観測点数を $n_i$ とする。

サイクル $i$ の $j$ 番目の観測を

- 角度: $\theta_{ij}$ [deg]
- センサ値: $y_{ij}$
- 時刻: $t_{ij}$

とする。

### 4.2 ラジアン角

三角関数に入れる角度はラジアンに直す。  
以降

$$
\vartheta = 2\pi \frac{\theta}{360}
$$

を使う。

### 4.3 窓長とストライド

窓長を $W$、ストライドを $S$ とする。

$$
W \in \mathbb{N},\quad W \ge 1
$$

$$
S \in \mathbb{N},\quad S \ge 1
$$

既定値は

$$
W = 10,\qquad S = 10
$$

とする。

### 4.4 窓数

サイクル列の長さが $N$ のとき、窓数 $H$ は

$$
H = \left\lfloor \frac{N-W}{S} \right\rfloor + 1
$$

である。

### 4.5 窓ラベル

1サイクルごとの異常ラベルを $\ell_i \in \{0,1\}$ とする。  
窓ラベルは

$$
\ell_h^{(W)} = \max_{i \in I_h} \ell_i
$$

とする。  
窓内に 1 サイクルでも異常があれば、その窓は異常とする。

---

## 5. データ分割仕様

### 5.1 分割の順序

データ分割は、先にサイクル列として行う。  
その後で、各群の内部だけで窓を作る。

つまり順序は

1. サイクル切り出し  
2. fit / threshold / test 分割  
3. 各群で窓生成

である。

これにより、fit と threshold をまたぐ窓、threshold と test をまたぐ窓が生じず、情報漏れを防げる。

### 5.2 分割比率

既定値は次の通りとする。

| 群 | 用途 | 既定比率 |
|---|---|---:|
| fit | 正常分布の学習 | 60% |
| threshold | しきい値設定 | 20% |
| test | 最終評価 | 20% |

### 5.3 時系列順分割

分割は原則として時間順またはサイクル順に前から後ろへ切る。  
ランダム分割は採用しない。

---

## 6. 窓ごとの調和回帰モデル

### 6.1 モデル式

窓 $h$ の代表周期モデルを

$$
\hat y_h(\vartheta)
=
a_{0,h}
+
\sum_{k=1}^{K}
\left(
a_{k,h}\cos(k\vartheta)
+
b_{k,h}\sin(k\vartheta)
\right)
$$

とする。

ここで $K$ は最大次数であり、第3版の既定値は

$$
K = 3
$$

とする。

### 6.2 観測方程式

窓 $h$ に含まれる各観測点 $(\theta_{hm}, y_{hm})$ に対して

$$
y_{hm} = \hat y_h(\vartheta_{hm}) + r_{hm}
$$

とする。  
$r_{hm}$ は窓モデルに対する観測残差である。

### 6.3 設計行列

窓 $h$ の設計行列 $X_h \in \mathbb{R}^{N_h \times (2K+1)}$ を

$$
X_h =
\begin{bmatrix}
1 & \cos \vartheta_{h1} & \sin \vartheta_{h1} & \cdots & \cos K\vartheta_{h1} & \sin K\vartheta_{h1}\\
1 & \cos \vartheta_{h2} & \sin \vartheta_{h2} & \cdots & \cos K\vartheta_{h2} & \sin K\vartheta_{h2}\\
\vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
1 & \cos \vartheta_{hN_h} & \sin \vartheta_{hN_h} & \cdots & \cos K\vartheta_{hN_h} & \sin K\vartheta_{hN_h}
\end{bmatrix}
$$

とする。

係数ベクトルを

$$
\beta_h =
\begin{bmatrix}
a_{0,h} & a_{1,h} & b_{1,h} & \cdots & a_{K,h} & b_{K,h}
\end{bmatrix}^{\top}
$$

とする。

### 6.4 係数推定

係数は最小二乗で求める。

$$
\hat\beta_h
=
\arg\min_{\beta}
\lVert y_h - X_h \beta \rVert_2^2
$$

実装では

$$
\hat\beta_h = X_h^{+} y_h
$$

または `lstsq` を用いる。

### 6.5 この形を採る理由

1サイクルあたり 9〜12 点と少なくても、10サイクル窓なら総観測点数はおよそ 90〜120 点になる。  
そのため、窓単位で回帰した方が係数推定が安定しやすい。

---

## 7. 窓代表波形の補間

### 7.1 密な角度グリッド

窓モデルから 1 周期の代表波形を復元するため、角度グリッド長を $L$ とし、

$$
\theta_g = 360 \frac{g}{L}, \qquad g=0,1,\dots,L-1
$$

を定義する。

ラジアンでは

$$
\vartheta_g = 2\pi \frac{g}{L}
$$

である。

既定値は

$$
L = 256
$$

とする。

### 7.2 補間波形

窓 $h$ の代表補間波形を

$$
x_h[g] = \hat y_h(\vartheta_g)
$$

とする。  
第3版では、この $x_h[g]$ を「窓 $h$ の 1 周期代表波形」と呼ぶ。

### 7.3 重要な意味

この補間波形は、窓内の 10 サイクルや 30 サイクルをそのまま連結した長い波形ではない。  
**窓内の全観測点を 0〜360 度の 1 周期上に重ねて作った、代表的な 1 周期波形**である。

---

## 8. 窓ごとに計算する基本量

第3版では、1サイクル特徴量を先に作らない。  
代わりに、窓ごとに調和回帰し、その窓から基本量を直接計算する。

### 8.1 次数成分の振幅

各次数 $k=1,\dots,K$ について

$$
A_{k,h} = \sqrt{a_{k,h}^2 + b_{k,h}^2}
$$

と定義する。

### 8.2 次数成分の位相

各次数 $k=1,\dots,K$ について

$$
\phi_{k,h} = \operatorname{atan2}(b_{k,h}, a_{k,h})
$$

と定義する。

### 8.3 補間波形の全体 RMS

補間波形の全体 RMS を

$$
\mathrm{RMS}_h = \sqrt{\frac{1}{L}\sum_{g=0}^{L-1} x_h[g]^2}
$$

とする。

### 8.4 観測残差

窓 $h$ の各観測点の残差を

$$
r_{hm} = y_{hm} - \hat y_h(\vartheta_{hm})
$$

とする。

### 8.5 残差 RMS

$$
\mathrm{RMS}^{(r)}_h
=
\sqrt{\frac{1}{N_h}\sum_{m=1}^{N_h} r_{hm}^2}
$$

とする。

### 8.6 残差の上位量

$$
P95_h
=
\operatorname{quantile}_{0.95}\left(|r_{h1}|,\dots,|r_{hN_h}|\right)
$$

$$
M_h = \max_m |r_{hm}|
$$

を定義する。

---

## 9. 窓特徴量

### 9.1 上下ずれ特徴量

上下ずれ段の特徴量は

$$
z_h^{(\mathrm{level})}
=
\begin{bmatrix}
a_{0,h} \\
\mathrm{RMS}_h
\end{bmatrix}
$$

とする。

### 9.2 振幅変化特徴量

振幅段の特徴量は

$$
z_h^{(\mathrm{amp})}
=
\begin{bmatrix}
A_{1,h} \\
A_{2,h} \\
A_{3,h}
\end{bmatrix}
$$

とする。

### 9.3 位相変化特徴量

fit 用正常窓から次数ごとの代表位相 $\bar\phi_k$ を求める。  
位相差は

$$
\delta_{k,h} = \operatorname{wrap}(\phi_{k,h} - \bar\phi_k)
$$

とする。ここで

$$
\operatorname{wrap}(x) = ((x+\pi)\bmod 2\pi)-\pi
$$

である。

位相段の特徴量は

$$
z_h^{(\mathrm{phase})}
=
\begin{bmatrix}
\delta_{1,h} \\
\delta_{2,h} \\
\delta_{3,h}
\end{bmatrix}
$$

とする。

### 9.4 低次周波数成分変化特徴量

次数配分を

$$
p_{k,h}
=
\frac{A_{k,h}^2}{A_{1,h}^2 + A_{2,h}^2 + A_{3,h}^2 + \varepsilon}
$$

とする。  
既定値として

$$
\varepsilon = 10^{-12}
$$

を用いる。

特徴量は

$$
z_h^{(\mathrm{freq})}
=
\begin{bmatrix}
p_{1,h} \\
p_{2,h} \\
p_{3,h}
\end{bmatrix}
$$

とする。

### 9.5 局所・高次残差異常特徴量

局所異常や高次側異常は、窓モデルで説明できなかった残差側で見る。  
特徴量は

$$
z_h^{(\mathrm{resid})}
=
\begin{bmatrix}
\mathrm{RMS}^{(r)}_h \\
P95_h \\
M_h
\end{bmatrix}
$$

とする。

---

## 10. 品質指標

### 10.1 目的

窓内観測が角度的に偏っていると、回帰係数は不安定になる。  
そのため、異常スコアとは別に、窓ごとの信頼度を見る品質指標を持つ。

### 10.2 総観測点数

$$
Q_h^{(\mathrm{nobs})} = N_h
$$

とする。

### 10.3 平均サイクル点数

$$
Q_h^{(\mathrm{nobs\_cycle})} = \frac{N_h}{W}
$$

とする。

### 10.4 窓角度カバー率

窓内全観測角度を昇順に並べたものを

$$
\theta^{\uparrow}_{h,(1)} \le \theta^{\uparrow}_{h,(2)} \le \cdots \le \theta^{\uparrow}_{h,(N_h)}
$$

とする。  
角度差列を

$$
g_{h,m} = \theta^{\uparrow}_{h,(m+1)} - \theta^{\uparrow}_{h,(m)} \quad (m=1,\dots,N_h-1)
$$

$$
g_{h,N_h} = 360 - \theta^{\uparrow}_{h,(N_h)} + \theta^{\uparrow}_{h,(1)}
$$

と定義する。

最大角度ギャップを

$$
G_h = \max_m g_{h,m}
$$

とし、窓角度カバー率を

$$
C_h = 1 - \frac{G_h}{360}
$$

とする。

### 10.5 使い方

品質指標は異常判定そのものには入れない。  
ただし $C_h$ が極端に低い場合や $N_h$ が極端に少ない場合は、低信頼として別表示する。

---

## 11. 正常分布の学習

### 11.1 学習対象

各監視段

$$
m \in \{\mathrm{level}, \mathrm{amp}, \mathrm{phase}, \mathrm{freq}, \mathrm{resid}\}
$$

に対し、fit 用正常窓から平均ベクトル $\mu_W^{(m)}$ とロバスト共分散行列 $\Sigma_W^{(m)}$ を学習する。

### 11.2 代表位相

fit 用正常窓の位相 $\phi_{k,h}$ に対して、次数ごとに円環平均を取り、代表位相 $\bar\phi_k$ を求める。

$$
\bar\phi_k
=
\operatorname{atan2}
\left(
\frac{1}{H_{\mathrm{fit}}}\sum_h \sin \phi_{k,h},
\frac{1}{H_{\mathrm{fit}}}\sum_h \cos \phi_{k,h}
\right)
$$

### 11.3 学習単位

第3版で学習するのは、1サイクル特徴量ではなく、**窓ごとの特徴量 $z_h^{(m)}$** である。  
ここが第2版との大きな違いである。

---

## 12. 各段の異常スコア

窓 $h$ の各段スコアは、窓特徴量に対するロバストなマハラノビス距離とする。

### 12.1 上下ずれ

$$
s_h^{(\mathrm{level})}
=
\left(z_h^{(\mathrm{level})}-\mu_W^{(\mathrm{level})}\right)^\top
\left(\Sigma_W^{(\mathrm{level})}\right)^{-1}
\left(z_h^{(\mathrm{level})}-\mu_W^{(\mathrm{level})}\right)
$$

### 12.2 振幅変化

$$
s_h^{(\mathrm{amp})}
=
\left(z_h^{(\mathrm{amp})}-\mu_W^{(\mathrm{amp})}\right)^\top
\left(\Sigma_W^{(\mathrm{amp})}\right)^{-1}
\left(z_h^{(\mathrm{amp})}-\mu_W^{(\mathrm{amp})}\right)
$$

### 12.3 位相変化

$$
s_h^{(\mathrm{phase})}
=
\left(z_h^{(\mathrm{phase})}-\mu_W^{(\mathrm{phase})}\right)^\top
\left(\Sigma_W^{(\mathrm{phase})}\right)^{-1}
\left(z_h^{(\mathrm{phase})}-\mu_W^{(\mathrm{phase})}\right)
$$

### 12.4 低次周波数成分変化

$$
s_h^{(\mathrm{freq})}
=
\left(z_h^{(\mathrm{freq})}-\mu_W^{(\mathrm{freq})}\right)^\top
\left(\Sigma_W^{(\mathrm{freq})}\right)^{-1}
\left(z_h^{(\mathrm{freq})}-\mu_W^{(\mathrm{freq})}\right)
$$

### 12.5 局所・高次残差異常

$$
s_h^{(\mathrm{resid})}
=
\left(z_h^{(\mathrm{resid})}-\mu_W^{(\mathrm{resid})}\right)^\top
\left(\Sigma_W^{(\mathrm{resid})}\right)^{-1}
\left(z_h^{(\mathrm{resid})}-\mu_W^{(\mathrm{resid})}\right)
$$

---

## 13. しきい値と総合判定

### 13.1 しきい値

threshold 用正常窓に対して各段スコアを計算し、上側分位点をしきい値とする。  
既定値は

$$
q_{\mathrm{thr}} = 0.995
$$

とする。

### 13.2 正規化スコア

各段の正規化スコアを

$$
u_{h,m} = \frac{s_h^{(m)}}{\tau_{W,m}}
$$

とする。

### 13.3 総合スコア

ユーザー要求は「どれか1本で危険なら異常」であるため、総合スコアは最大値とする。

$$
u_{h,\mathrm{total}}
=
\max\left(
u_{h,\mathrm{level}},
u_{h,\mathrm{amp}},
u_{h,\mathrm{phase}},
u_{h,\mathrm{freq}},
u_{h,\mathrm{resid}}
\right)
$$

### 13.4 判定

$$
u_{h,\mathrm{total}} > 1
$$

なら異常とする。

### 13.5 主因

$$
\operatorname{cause}_h
=
\arg\max_{m}
u_{h,m}
$$

とする。

---

## 14. 可視化仕様

### 14.1 主画面

主画面は窓番号または窓右端サイクル番号を横軸にした時系列グラフとする。  
表示する系列は

- 総合スコア
- 上下ずれ
- 振幅変化
- 位相変化
- 低次周波数成分変化
- 局所・高次残差異常

の 6 本とする。

### 14.2 横軸

窓 $h$ の右端サイクル番号は

$$
i_h^{(\mathrm{right})} = (h-1)S + W
$$

とする。  
既定ではこれを横軸位置とする。

### 14.3 補助表示

補助として、窓ごとの代表補間波形 $x_h[g]$ を表示できるようにする。  
異常窓では「正常代表波形」と「その窓の代表波形」を重ねると原因説明がしやすい。

---

## 15. 評価仕様

### 15.1 必須評価

必須評価は窓単位で行う。  
最低限、混同行列、再現率、適合率、F1、PR-AUC を出す。

### 15.2 ラベル対応

窓ラベルには $\ell_h^{(W)}$ を使う。  
したがって、評価単位も窓である。

### 15.3 $W=1$ との比較

設計妥当性の確認として、$W=1$ と $W=10$ 以上を並べて比較することを推奨する。  
これにより、窓化によってどの程度安定化したかを確認できる。

---

## 16. 実装手順

### 16.1 入力整形

連続データを時間順に並べ、角度の折り返しからサイクル境界を推定し、各サイクルに分ける。

### 16.2 分割

正常サイクル列を fit / threshold / test-normal に時系列順で分ける。  
異常サイクル列は test-anomaly とする。  
その後、各群の内部で窓を作る。

### 16.3 窓データ作成

窓 $h$ に対して、窓内サイクルの全観測点を結合し、$(\theta_{hm}, y_{hm}, t_{hm})$ を作る。

### 16.4 窓調和回帰

窓ごとに設計行列 $X_h$ を作り、最小二乗で $\hat\beta_h$ を求める。

### 16.5 補間波形生成

密な角度グリッド上で $x_h[g]=\hat y_h(\vartheta_g)$ を計算する。

### 16.6 特徴量計算

$A_{k,h}$、$\phi_{k,h}$、$\mathrm{RMS}_h$、$\mathrm{RMS}^{(r)}_h$、$P95_h$、$M_h$ を計算し、各段の窓特徴量を作る。

### 16.7 学習

fit 用正常窓から代表位相、平均ベクトル、ロバスト共分散を学習する。

### 16.8 しきい値

threshold 用正常窓から各段スコアの上側 99.5 パーセンタイルを求める。

### 16.9 test

test 窓に対して各段スコア、正規化スコア、総合スコア、主因を計算する。

---



## 17. 実装方法（Python ライブラリを用いた実装指針）

本章では、第3版の処理を Python の既存ライブラリでどう実装するかを明確にする。

### 17.1 使用ライブラリ

推奨ライブラリは次の通りとする。

| 用途 | 推奨ライブラリ | 主な関数・クラス |
|---|---|---|
| 配列計算 | `numpy` | `np.array`, `np.column_stack`, `np.sqrt`, `np.quantile`, `np.max` |
| 表形式管理 | `pandas` | `DataFrame`, `groupby`, `concat` |
| 最小二乗 | `numpy.linalg` | `np.linalg.lstsq`, `np.linalg.pinv` |
| ロバスト共分散 | `sklearn.covariance` | `MinCovDet` |
| 異常スコア | `sklearn.covariance` | `MinCovDet.mahalanobis` |
| 評価 | `sklearn.metrics` | `confusion_matrix`, `classification_report`, `precision_recall_curve`, `average_precision_score` |
| 可視化 | `matplotlib` | `plt.plot`, `plt.scatter`, `plt.axhline` |

### 17.2 窓単位の調和回帰の実装

1サイクルごとではなく、**窓ごとの全観測点をまとめて回帰する**ことが重要である。

```python
import numpy as np

def build_design_matrix(theta_deg: np.ndarray, K: int = 3) -> np.ndarray:
    # 度 -> ラジアン
    theta_rad = 2.0 * np.pi * theta_deg / 360.0
    # 定数項（平均の高さ）
    cols = [np.ones_like(theta_rad)]
    # cos, sin の成分を順番に追加
    for k in range(1, K + 1):
        cols.append(np.cos(k * theta_rad))
        cols.append(np.sin(k * theta_rad))
    # 横に並べて行列にする
    return np.column_stack(cols)

def fit_window_harmonic(theta_deg: np.ndarray, y: np.ndarray, K: int = 3) -> np.ndarray:
    # 角度から説明行列 X を作る
    X = build_design_matrix(theta_deg, K=K)
    # X @ beta ≈ y となる beta を最小二乗で求める
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    # 求まった係数を返す
    return beta
```

### 17.3 補間波形の生成

```python
def predict_harmonic_on_grid(beta: np.ndarray, L: int = 256, K: int = 3):
    # 0〜360度未満の等間隔グリッドを作る
    grid_deg = np.arange(L) * 360.0 / L
    # その角度に対する説明行列を作る
    Xg = build_design_matrix(grid_deg, K=K)
    # 予測波形を計算
    y_grid = Xg @ beta
    return grid_deg, y_grid
```

### 17.4 窓特徴量の計算

```python
def harmonic_amp_phase(beta: np.ndarray, K: int = 3):
    a0 = beta[0]
    amps = []
    phases = []
    for k in range(K):
        a = beta[1 + 2*k]
        b = beta[2 + 2*k]
        amps.append(np.sqrt(a*a + b*b))
        phases.append(np.arctan2(b, a))
    return a0, np.asarray(amps), np.asarray(phases)

def residual_features(theta_deg: np.ndarray, y: np.ndarray, beta: np.ndarray, L: int = 256, K: int = 3):
    X = build_design_matrix(theta_deg, K=K)
    y_hat_obs = X @ beta
    r = y - y_hat_obs

    _, y_grid = predict_harmonic_on_grid(beta, L=L, K=K)
    rms_grid = np.sqrt(np.mean(y_grid**2))
    rms_resid = np.sqrt(np.mean(r**2))
    p95 = np.quantile(np.abs(r), 0.95)
    rmax = np.max(np.abs(r))
    return rms_grid, rms_resid, p95, rmax
```

### 17.5 位相差の実装

```python
def circular_mean(phases: np.ndarray) -> float:
    return np.arctan2(np.mean(np.sin(phases)), np.mean(np.cos(phases)))

def wrap_angle(x):
    return (x + np.pi) % (2.0 * np.pi) - np.pi
```

fit 用正常窓の位相から `circular_mean` で代表位相を求め、各窓で `wrap_angle(phi - phi_ref)` を計算する。

### 17.6 異常スコアの実装

第3版で異常検知スコアを出す場所は、各窓特徴量に対して `MinCovDet` を適用する部分である。

```python
from sklearn.covariance import MinCovDet

def fit_mcd(Z_fit: np.ndarray) -> MinCovDet:
    model = MinCovDet(random_state=0).fit(Z_fit)
    return model

def mahal_score(model: MinCovDet, Z: np.ndarray) -> np.ndarray:
    return model.mahalanobis(Z)
```

この `mahalanobis(...)` が、各段の異常スコア

$$
s_h^{(m)}
$$

を実際に計算する場所である。

### 17.7 しきい値の実装

threshold 用正常窓スコアから分位点を取る。

```python
def calc_threshold(scores_thr: np.ndarray, q: float = 0.995) -> float:
    return float(np.quantile(scores_thr, q))
```

### 17.8 総合スコアと主因

```python
def total_score_and_cause(u_level, u_amp, u_phase, u_freq, u_resid):
    U = np.column_stack([u_level, u_amp, u_phase, u_freq, u_resid])
    u_total = U.max(axis=1)
    cause_idx = U.argmax(axis=1)
    cause_names = np.array(["level", "amp", "phase", "freq", "resid"])
    cause = cause_names[cause_idx]
    return u_total, cause
```

### 17.9 評価の実装

```python
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score,
)

def evaluate_binary(y_true, u_total):
    y_pred = (u_total > 1.0).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True)
    ap = average_precision_score(y_true, u_total)
    pr, rc, thr = precision_recall_curve(y_true, u_total)
    return cm, report, ap, pr, rc, thr
```

---

## 18. 第3版で明確に答えた点

### 18.1 どこが第2版と違うのか

第2版は 1サイクル特徴量を先に作って窓で集約していた。  
第3版は、先に窓を作り、窓内全観測点で調和回帰し、その窓の代表波形から特徴量を出す。

### 18.2 1サイクルと同じ特徴量計算とは何か

意味は、「窓ごとに得られた 1 周期代表波形に対して、1サイクル版と同じ種類の特徴量を計算する」ということである。  
つまり、入力単位が 1サイクルから窓に変わるだけで、特徴量の意味づけは変えない。

### 18.3 何がうれしいのか

1サイクルあたりの観測点が少なくても、窓で束ねれば回帰が安定しやすい。  
その一方で、最終的には 1 周期の代表波形に戻すため、説明がしやすい。

---

## 19. 固定パラメータ一覧

| 項目 | 記号 | 既定値 |
|---|---:|---:|
| 最大次数 | $K$ | 3 |
| 窓長 | $W$ | 10 |
| ストライド | $S$ | 1 |
| 補間グリッド長 | $L$ | 256 |
| 安定化項 | $\varepsilon$ | $10^{-12}$ |
| しきい値分位点 | $q_{\mathrm{thr}}$ | 0.995 |
| 分割比率 | fit / threshold / test | 60 / 20 / 20 |

---

## 20. 参考文献

1. Lomb, N. R. (1976). Least-squares frequency analysis of unequally spaced data. *Astrophysics and Space Science*.  
2. Zechmeister, M., & Kürster, M. (2009). The generalised Lomb-Scargle periodogram. *Astronomy & Astrophysics*.  
3. Brüel & Kjær. *Order Tracking Analysis*.  
4. Rousseeuw, P. J., & Van Driessen, K. (1999). A fast algorithm for the minimum covariance determinant estimator. *Technometrics*.  
5. scikit-learn Documentation: robust covariance / MinCovDet.  
6. NIST/SEMATECH e-Handbook of Statistical Methods: Multivariate Methods and Control Charts.

本仕様における不等間隔周期データの最小二乗周期モデル、ロバスト距離による異常判定、円環量の平均、および窓単位の安定化の考え方は、上記の文献・資料に沿って整理している。
